# Generate clinical note embeddings (GPU, GCP)

Self-contained driver for `generate_clinical_embeddings.py` — runs on a GCP GPU VM/notebook,
**not** the cluster. Deliberately does not import `config.py` (that module hardcodes cluster
paths); the underlying script only depends on torch/numpy/transformers/tqdm.

Steps: install deps, verify a GPU is visible, set `TOKEN_PATH`/`EMBED_PATH`, pull tokenized
note batches from GCS, run the embedding script, push results back to GCS.

In [ ]:
%pip install -q torch numpy transformers tqdm zstandard

## Verify GPU

In [ ]:
import subprocess

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
assert torch.cuda.is_available(), "No CUDA device visible - check the GCP VM/notebook GPU runtime."
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Configure paths

`generate_clinical_embeddings.py` reads `CLINICAL_EMBED_GCP_ROOT` / `TOKEN_PATH` / `EMBED_PATH`
from the environment (defaults under `~/generate_clinical_embeddings`). Set them explicitly here
so the `gsutil` pull/push below targets the same directories.

In [ ]:
import os
from pathlib import Path

GCS_BUCKET = "gs://<your-bucket>/clinical_text_embedding_project"

GCP_ROOT = Path(os.environ.get("CLINICAL_EMBED_GCP_ROOT", str(Path.home() / "generate_clinical_embeddings")))
os.environ["CLINICAL_EMBED_GCP_ROOT"] = str(GCP_ROOT)
os.environ["TOKEN_PATH"] = str(GCP_ROOT / "tokens")
os.environ["EMBED_PATH"] = str(GCP_ROOT / "embeddings")

Path(os.environ["TOKEN_PATH"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["EMBED_PATH"]).mkdir(parents=True, exist_ok=True)

print(f"GCP root:   {GCP_ROOT}")
print(f"TOKEN_PATH: {os.environ['TOKEN_PATH']}")
print(f"EMBED_PATH: {os.environ['EMBED_PATH']}")

## Pull tokenized note batches from GCS

Produced by `run_preprocessing.ipynb` on the cluster and shipped here via `gsutil rsync`.

In [ ]:
pull_cmd = ["gsutil", "-m", "rsync", "-r", f"{GCS_BUCKET}/tokens", os.environ["TOKEN_PATH"]]
print(" ".join(pull_cmd))
subprocess.run(pull_cmd, check=True)

## Run embedding generation

In [ ]:
import sys

GCP_ROOT_SCRIPT_DIR = GCP_ROOT / "src"  # place generate_clinical_embeddings.py here, or adjust
SCRIPT_PATH = GCP_ROOT_SCRIPT_DIR / "generate_clinical_embeddings.py"

if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        f"generate_clinical_embeddings.py not found at {SCRIPT_PATH}. "
        "Copy it up from v2/pipelines/preprocessing/ before running this cell."
    )

subprocess.run([sys.executable, str(SCRIPT_PATH)], check=True)

## Push embeddings back to GCS

`knit_embeddings.py` (run on the cluster, see
[build_prediction_datasets.ipynb](build_prediction_datasets.ipynb)) reads from `EMBEDS_PATH`,
which should be synced from this bucket path.

In [ ]:
push_cmd = ["gsutil", "-m", "rsync", "-r", os.environ["EMBED_PATH"], f"{GCS_BUCKET}/embeddings"]
print(" ".join(push_cmd))
subprocess.run(push_cmd, check=True)
print("Done. Pull these down on the cluster into $BATCHED_DATA_PATH/embeddings before running build_prediction_datasets.ipynb.")